# 12 · Langfuse — open-source observability
Callback + `@observe`, framework-agnostic. Open https://cloud.langfuse.com → Traces.

In [ ]:
# Bootstrap: make the repo root importable so `import config` works from notebooks/
import sys, os
sys.path.insert(0, os.path.abspath(".."))
from config import assert_key
assert_key()
print("Gateway ready.")

In [ ]:
from config import get_langchain_llm
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langfuse import observe, get_client
from langfuse.langchain import CallbackHandler

handler = CallbackHandler()   # reads LANGFUSE_* from .env
chain = ChatPromptTemplate.from_template("Give 3 risks of deploying agents in prod about {topic}.") | get_langchain_llm() | StrOutputParser()
print(chain.invoke({"topic": "tool-calling agents"}, config={"callbacks": [handler]}))

@observe()
def scored_pipeline(topic: str) -> str:
    out = chain.invoke({"topic": topic}, config={"callbacks": [handler]})
    get_client().score_current_trace(name="length_ok", value=1 if len(out) < 800 else 0)
    return out

scored_pipeline("multi-agent orchestration")
get_client().flush()
print("Open Langfuse → Traces.")

## 🧪 Your turn
1. Add a **second** custom score, e.g. `contains_number` (1 if the output has a digit).
2. Re-run and compare both scores on the trace in the Langfuse UI.

In [ ]:
# your code here